# Perceptron & the XOR Problem

CSCI 6379 · Topic 13. A single neuron (perceptron) draws only a straight boundary, so it cannot solve XOR. Adding one hidden layer (an MLP) bends the boundary and solves it.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# XOR: output 1 when inputs differ, 0 when they are the same
X = torch.tensor([[0,0], [0,1], [1,0], [1,1]], dtype=torch.float32)
y = torch.tensor([[0],   [1],   [1],   [0]],   dtype=torch.float32)

## A single perceptron gets stuck

One neuron = one straight cut. XOR is not linearly separable, so the loss flattens at 0.25 (equivalent to outputting 0.5 everywhere) and never improves.

In [ ]:
class SingleLayerPerceptron(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(2, 1)          # a single straight cut
    def forward(self, x):
        return torch.sigmoid(self.fc(x))

torch.manual_seed(0)
model = SingleLayerPerceptron()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

for epoch in range(10000):
    optimizer.zero_grad()
    loss = criterion(model(X), y)
    loss.backward()
    optimizer.step()

print("final loss:", round(loss.item(), 4))            # -> 0.25, stuck
print("predictions:", model(X).round().flatten().tolist())

## An MLP with one hidden layer solves XOR

Two neurons in a hidden layer let the network bend the boundary. The loss drops to ~0 and the predictions become [0, 1, 1, 0]. (Plain SGD on this tiny problem is init-sensitive — it can stall in a local minimum; Adam converges reliably here.)

In [ ]:
class XOR_MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 2)     # input -> hidden layer (2 neurons)
        self.fc2 = nn.Linear(2, 1)     # hidden -> output
    def forward(self, x):
        x = torch.sigmoid(self.fc1(x))  # hidden layer + activation
        x = torch.sigmoid(self.fc2(x))  # output layer + activation
        return x

torch.manual_seed(4)
model = XOR_MLP()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.1)

for epoch in range(3000):
    optimizer.zero_grad()
    loss = criterion(model(X), y)
    loss.backward()
    optimizer.step()

print("final loss:", round(loss.item(), 4))            # -> ~0.0
print("raw outputs:", [round(v, 3) for v in model(X).flatten().tolist()])
print("predictions:", model(X).round().flatten().tolist())   # -> [0, 1, 1, 0]

## Visualize the decision boundaries

The perceptron leaves a flat sheet at 0.5; the MLP carves a bent, banded region that separates the two diagonals.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

gx, gy = np.meshgrid(np.linspace(-0.3, 1.3, 200), np.linspace(-0.3, 1.3, 200))
grid = torch.tensor(np.c_[gx.ravel(), gy.ravel()], dtype=torch.float32)
with torch.no_grad():
    P = model(grid).numpy().reshape(gx.shape)     # the trained MLP

plt.contourf(gx, gy, P, levels=20, cmap="RdBu_r", alpha=0.7)
plt.contour(gx, gy, P, levels=[0.5], colors="k", linewidths=2)
for (px, py), lab in zip([[0,0],[0,1],[1,0],[1,1]], [0,1,1,0]):
    plt.scatter(px, py, s=200, color=("red" if lab else "blue"), edgecolor="w", zorder=3)
plt.title("MLP decision boundary for XOR"); plt.xlabel("x1"); plt.ylabel("x2")
plt.show()